Nazario dataset exploration, structures the emails into a format that the ML models can use

Run on python 3.13.13 kernel, on Visual studios code, using juypter notebook
Generates "2_Naz_ml_ready.csv" which contains a label column and a text column, comprising of email subject + email body.
runs on the provided Nazario.csv file

In [1]:
import pandas as pd
#reading the dataset and viewing the first few rows
df = pd.read_csv("Nazario.csv")
print(df.head())
print(df.columns)
print(df.shape)

                                              sender  \
0  Mail System Internal Data <MAILER-DAEMON@monke...   
1                        cPanel <service@cpanel.com>   
2    Microsoft Outlook <recepcao@unimedceara.com.br>   
3                     Ann Garcia <AnGarcia@mcoe.org>   
4                 "USAA" <usaaacctupdate@sccu4u.com>   

                                 receiver  \
0                                     NaN   
1                         jose@monkey.org   
2                                     NaN   
3     "info@maaaaa.org" <info@maaaaa.org>   
4  Recipients <usaaacctupdate@sccu4u.com>   

                                    date  \
0             28 Sep 2017 09:57:25 -0400   
1        Fri, 30 Oct 2015 00:00:48 -0500   
2  Fri, 30 Oct 2015 06:21:59 -0300 (BRT)   
3        Fri, 30 Oct 2015 14:54:33 +0000   
4        Fri, 30 Oct 2015 14:02:33 -0500   

                                             subject  \
0  DON'T DELETE THIS MESSAGE -- FOLDER INTERNAL DATA   
1              

In [2]:
# Remove first row as its non email content
df = df.iloc[1:].reset_index(drop=True)

# Keep only the columns needed
df = df[["subject", "body"]]
#merging the subject and body
df["text"] = df["subject"].fillna("") + " " + df["body"].fillna("")
#changing the label default text
df["label"] = "phishing"
#keeping the only 2 columns needed
df = df[["text", "label"]]


print(df.head())
print(df.columns)
print(df.shape)

                                                text     label
0  Verify Your Account Business with  \t\t\t\t\t\...  phishing
1  Helpdesk Mailbox Alert!!! Your two incoming ma...  phishing
2  IT-Service Help Desk Password will expire in 3...  phishing
3  Final USAA Reminder - Update Your Account Now ...  phishing
4  =?utf-8?Q?Dear=20Client=20=3a=20Update=20Your=...  phishing
Index(['text', 'label'], dtype='object')
(1564, 2)


In [3]:
import quopri
import re
import csv
from email.header import decode_header

#to see before and after
print(df.head())

def clean_text(text):
    if pd.isna(text):
        #if email row is emptry return emptry string to prevent errors
        return ""
    #turn all into plain python string
    text = str(text)

    #Encoding normalization
    try:
        text = text.encode("latin1", errors="ignore").decode("utf-8", errors="ignore")
    except:
        pass

    #set to quoted printable that is a email-safe encoding
    try:
        text = quopri.decodestring(text).decode("utf-8", errors="ignore")
    except:
        pass

    #remove MIME utf-8 artifacts and hidden line breaks
    text = re.sub(r"=\?utf-8\?Q\?.*?\?=", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"=\?utf-8\?Q\?", " ", text, flags=re.IGNORECASE)
    #normalize white space
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

#applying clean
df["text"] = df["text"].apply(clean_text)
# remove empty rows
df = df[df["text"] != ""].reset_index(drop=True)
print(df.head())

#print the cleaned data
df.to_csv(
    "2_Naz_ml_ready.csv",
    index=False,
    quoting=csv.QUOTE_ALL,
    escapechar="\\"
)



                                                text     label
0  Verify Your Account Business with  \t\t\t\t\t\...  phishing
1  Helpdesk Mailbox Alert!!! Your two incoming ma...  phishing
2  IT-Service Help Desk Password will expire in 3...  phishing
3  Final USAA Reminder - Update Your Account Now ...  phishing
4  =?utf-8?Q?Dear=20Client=20=3a=20Update=20Your=...  phishing
                                                text     label
0  Verify Your Account Business with cPanel & WHM...  phishing
1  Helpdesk Mailbox Alert!!! Your two incoming ma...  phishing
2  IT-Service Help Desk Password will expire in 3...  phishing
3  Final USAA Reminder - Update Your Account Now ...  phishing
4  PayPal Secure Dear Client, We have noticed tha...  phishing
